# 🛰️ Building Age Classification — Live JupyterLab Demo
**START HERE — run cells in order, top to bottom.**

This notebook runs our **real, saved Stage 2 submission code**, not a fake demonstration.
We give the model a few Amsterdam pixels whose age classes are known (the **support set**).
It then predicts classes for **different, unlabeled query pixels**.

| Output class | Amsterdam construction period |
|---|---|
| 1 | Up to and including 1945 |
| 2 | 1946–1984 |
| 3 | 1985–2004 |
| 4 | 2005 onwards |

### What this package can and cannot do
✅ Reads **prepared satellite features**, adapts to Amsterdam, predicts classes 1–4.  
✅ Tests the final saved Madrid model bundle and Stage 2 entry script.  
❌ Does **not** accept arbitrary JPG/PNG satellite images or raw Parquet directly.  
❌ The 72.32% Macro F1 is an **Amsterdam development result**, not a hidden-test score.

**Keep `demo_inputs/` private:** it will contain labeled Amsterdam *support* pixels. They are for a private presentation, not part of the final submission ZIP.


## Step 1 — Find the actual trained models and server files
We do **not** download or rebuild the 252 MB models. The server already has the verified saved bundle.
This cell checks that the models, feature files and our entry-point script exist.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 1: VERIFY PATHS AND SETUP
# ============================================================
from pathlib import Path
import sys, subprocess, json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()               # Where this notebook is running
ROOT = Path.home() / 'hackathon_results'      # Existing hosted JupyterLab data
DEST = ROOT / 'presentation_day_submission_20261118'
BUNDLE = DEST / 'model_bundle'

BASE = ROOT / 'exp013h_102feature_candidate.npz'
ANNUAL = ROOT / 'exp017c_amsterdam_annual_history/amsterdam_annual_252features.npz'
REF_TRIAL = ROOT / 'exp017f_random_support_354/trial_checkpoints/trial_01.npz'
BACKUP = ROOT / 'private_recovery/verified_fixed_data_backup.npz'
REF_TEN = ROOT / 'exp017f_random_support_354/lgbm102_vs354_10trials.npz'

required = {
    'run_stage2.py (entry point)': PROJECT / 'run_stage2.py',
    'predictor.py (prediction engine)': PROJECT / 'age_hackathon/predictor.py',
    'model SHA manifest': BUNDLE / 'manifest.json',
    'Madrid Random Forest': BUNDLE / 'madrid_rf60.joblib',
    'Madrid LightGBM': BUNDLE / 'madrid_lgbm354.joblib',
    'model feature schema': BUNDLE / 'schema.npz',
    'Amsterdam 102-feature archive': BASE,
    'Amsterdam annual feature archive': ANNUAL,
    'saved held-out trial': REF_TRIAL,
    'PRIVATE support-label archive': BACKUP,
}
print('Project directory:', PROJECT)
print('Server data root:', ROOT)
for label, path in required.items():
    print(('OK     ' if path.is_file() else 'MISSING'), label)

missing = [str(p) for p in required.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Some required HOSTED-SERVER files are missing.\n' +
        '\n'.join(missing) +
        '\nDo not retrain or overwrite saved experiments. '
        'See README.md for where the trusted saved files belong.'
    )

sys.path.insert(0, str(PROJECT))
for name in ('demo_inputs', 'demo_outputs'):
    (PROJECT / name).mkdir(exist_ok=True)
with (BUNDLE / 'manifest.json').open(encoding='utf-8') as f:
    manifest = json.load(f)
print('\nSaved model mixture:', manifest.get('mixture', 'see manifest.json'))
print('All required demonstration files: PRESENT')


## Step 2 — Open verified Amsterdam satellite features

We are **not** feeding photographs into the model. `X102` gives 102 engineered values per pixel;
`X252` gives six spectral bands over 42 years. The LightGBM receives all **354 columns**.

We reuse the *exact saved support/query split* from our 200-shot development trial 1.
The 800 support pixels are separate from the 25,192 query pixels. Query labels are **not used to build model inputs or adapt the model**.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 2: LOAD FEATURES AND SAVED SPLIT
# ============================================================
with np.load(BASE, allow_pickle=False) as d:
    X102 = d['X_amsterdam'].copy()
    pixel_ids = d['pixel_ids_amsterdam'].copy()
with np.load(ANNUAL, allow_pickle=False) as d:
    X252 = d['features'].copy()
    np.testing.assert_array_equal(pixel_ids, d['pixel_ids'])
with np.load(REF_TRIAL, allow_pickle=False) as d:
    support_idx = d['support_indices'].copy()
    query_idx = d['query_indices'].copy()
    reference_200 = d['predictions'][1].copy()

# Read only SUPPORT labels at this stage. No query labels enter inference.
with np.load(BACKUP, allow_pickle=False) as d:
    support_y = d['y_amsterdam'][support_idx].copy()

assert X102.shape == (25992, 102), X102.shape
assert X252.shape == (25992, 252), X252.shape
assert len(support_idx) == 800
assert len(query_idx) == 25192
assert reference_200.shape == (len(query_idx),)
assert len(np.unique(np.concatenate((support_idx, query_idx)))) == len(X102)
assert all((support_y == cls).sum() == 200 for cls in (1, 2, 3, 4))

print('Amsterdam pixels:', len(X102))
print('Engineered feature columns:', X102.shape[1])
print('Annual spectral columns:', X252.shape[1])
print('Labeled support pixels:', len(support_idx), '(200 per class)')
print('Unlabeled prediction targets:', len(query_idx))
print('Support and query overlap: NONE')
print('Feature alignment: VERIFIED')


## Step 3 — Run the actual `run_stage2.py` entry point

Change **`N_SHOTS`** to `5`, `25`, `50`, `100` or `200` if judges ask for another budget.
`200` reproduces archived development trial 1; for smaller budgets we use the first `n` support rows per class from that same trial (a demonstration, **not** an archived official benchmark).

The CLI gets the *support labels*, but it gets **no query labels**. It adapts the LDA and (when `n≥25`) continues the LightGBM. Then it predicts all query pixels.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 3: ACTUAL PYTHON ENTRY SCRIPT
# ============================================================
from datetime import datetime, timezone

N_SHOTS = 200  # Allowed: 5, 25, 50, 100, 200
assert N_SHOTS in (5, 25, 50, 100, 200)

if N_SHOTS == 200:
    # Keep archived 200-shot training order exactly unchanged.
    selected = np.arange(len(support_y))
else:
    selected = np.concatenate([
        np.flatnonzero(support_y == cls)[:N_SHOTS]
        for cls in (1, 2, 3, 4)
    ])

selected_support = support_idx[selected]
selected_labels = support_y[selected]
assert len(selected_support) == 4 * N_SHOTS
assert all((selected_labels == cls).sum() == N_SHOTS for cls in (1, 2, 3, 4))

# This PRIVATE input NPZ contains labeled support rows, never query labels.
input_file = PROJECT / 'demo_inputs' / f'demo_stage2_{N_SHOTS}shots.npz'
np.savez_compressed(
    input_file,
    support102=X102[selected_support],
    support252=X252[selected_support],
    support_labels=selected_labels,
    query102=X102[query_idx],
    query252=X252[query_idx],
    query_pixel_ids=pixel_ids[query_idx]
)

# A unique filename prevents overwriting any previous demo output.
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output_file = PROJECT / 'demo_outputs' / f'predictions_{N_SHOTS}shots_{stamp}.npz'

command = [
    sys.executable, str(PROJECT / 'run_stage2.py'),
    '--bundle', str(BUNDLE),
    '--input', str(input_file),
    '--n', str(N_SHOTS),
    '--output', str(output_file)
]
print('Running the ACTUAL Stage 2 entry point...')
print('Command:', ' '.join(command))
print('This adapts on support labels only. It may take some time.\n', flush=True)
subprocess.run(command, cwd=str(PROJECT), check=True)

with np.load(output_file, allow_pickle=False) as d:
    predictions = d['predictions'].copy()
    np.testing.assert_array_equal(d['query_pixel_ids'], pixel_ids[query_idx])
assert predictions.shape == (len(query_idx),)
assert np.isin(predictions, (1, 2, 3, 4)).all()
print('\nREAL INFERENCE FINISHED:', len(predictions), 'query predictions')
print('Saved predictions to:', output_file)
if N_SHOTS == 200:
    np.testing.assert_array_equal(predictions, reference_200)
    print('EXACT MATCH to saved 017F trial-1 predictions: PASSED')


## Step 4 — Show 20 actual predictions to the judges
Each row is **one satellite-data pixel**, not an uploaded photograph. The class is a *predicted age category*.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 4: EASY-TO-READ PREDICTION TABLE
# ============================================================
AGE_PERIODS = {
    1: '1945 or earlier', 2: '1946–1984',
    3: '1985–2004', 4: '2005 or later'
}
first = 20
sample_ids = pixel_ids[query_idx[:first]]
table_data = {
    'Pixel number': np.arange(1, first + 1),
    'Predicted class': predictions[:first],
    'Estimated construction period': [AGE_PERIODS[int(c)] for c in predictions[:first]],
}
if sample_ids.ndim == 2 and sample_ids.shape[1] >= 2:
    table_data['Pixel X'] = sample_ids[:, 0]
    table_data['Pixel Y'] = sample_ids[:, 1]
else:
    table_data['Pixel ID'] = sample_ids.astype(str)
display(pd.DataFrame(table_data))
print('Predicted class distribution across ALL query pixels:')
display(pd.DataFrame({
    'Class': [1, 2, 3, 4],
    'Predicted pixels': [int(np.sum(predictions == c)) for c in (1, 2, 3, 4)]
}))


## Step 5 — Check accuracy *after* the prediction is finished

**Only here** do we reveal query labels to calculate Macro F1 and a confusion matrix.
These labels are for *evaluation*; they are never passed into `run_stage2.py` or the adaptation process.
This is a **development split**, not the secret competition test.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 5: EVALUATE WITHOUT TRAINING LEAKAGE
# ============================================================
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Read labels only AFTER the saved predictions already exist.
with np.load(BACKUP, allow_pickle=False) as d:
    true_query = d['y_amsterdam'][query_idx].copy()

classes = [1, 2, 3, 4]
f1_each = f1_score(true_query, predictions, labels=classes, average=None, zero_division=0)
macro_f1 = float(f1_each.mean())
score_table = pd.DataFrame({
    'Age class': classes,
    'Construction period': [AGE_PERIODS[c] for c in classes],
    'F1 score': np.round(f1_each, 4)
})
print(f'Actual THIS-TRIAL {N_SHOTS}-shot Macro F1: {macro_f1:.4f}')
display(score_table)

cm = confusion_matrix(true_query, predictions, labels=classes)
fig, ax = plt.subplots(figsize=(6.5, 5.2))
ConfusionMatrixDisplay(cm, display_labels=['C1', 'C2', 'C3', 'C4']).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d'
)
ax.set_title(f'Amsterdam development — {N_SHOTS} shots/class (one trial)')
plt.tight_layout()
figfile = PROJECT / 'demo_outputs' / f'confusion_{N_SHOTS}shots_{stamp}.png'
fig.savefig(figfile, dpi=150)
plt.show()
print('Confusion matrix saved:', figfile)
print('NOTE: Macro F1 is not the same as classification accuracy.')


## Step 6 — Show the verified ten-trial result (without retraining)

The historical Experiment 017F saved ten different 200-shot support trials. We report their **mean and spread**,
but the query pixels overlap, so these are **not ten independent held-out datasets**.


In [ ]:
# ============================================================
# LIVE DEMO — STEP 6: READ PREVIOUSLY VERIFIED 10-TRIAL SCORES
# ============================================================
if REF_TEN.is_file():
    with np.load(REF_TEN, allow_pickle=False) as d:
        saved_scores = d['scores'].copy()
    assert saved_scores.shape == (10, 2), saved_scores.shape
    trial_scores = saved_scores[:, 1]
    average = float(trial_scores.mean())
    spread = float(trial_scores.std(ddof=0))
    print('Existing 200-shot Amsterdam development experiment:')
    print('10-trial mean Macro F1:', f'{average:.4f}')
    print('Population SD:', f'{spread:.4f}')
    print('Scores:', np.round(trial_scores, 4))
    print('Interpretation: development evidence, NOT official hidden-test performance.')
else:
    print('Optional saved 10-trial score file not present:', REF_TEN)
    print('Previously verified development mean: 0.7232 (10 trials, 200 shots/class)')


## Optional Step 7 — Stage 1 (Madrid-trained model, without adaptation)
This shows that the already-trained **Madrid LightGBM354** can make direct predictions.
For convenience the example below applies it to the same Amsterdam feature rows **without Amsterdam labels**;
therefore it is a **zero-shot demonstration**, *not* a Madrid validation score.


In [ ]:
# ============================================================
# LIVE DEMO — OPTIONAL STEP 7: MADRID MODEL ZERO-SHOT
# ============================================================
from age_hackathon import load_bundle

madrid_model = load_bundle(BUNDLE)  # Includes SHA-256 integrity verification
predicted_stage1, probabilities_stage1 = madrid_model.predict_stage1(
    X102[query_idx[:20]], X252[query_idx[:20]]
)
assert probabilities_stage1.shape == (20, 4)
display(pd.DataFrame({
    'Pixel': np.arange(1, 21),
    'Madrid model (no Amsterdam adaptation)': predicted_stage1,
    'Stage 2 adapted hybrid': predictions[:20]
}))
print('Madrid-trained Stage 1 model loaded and ran successfully.')
print('Different predictions are possible because Stage 2 adapts to Amsterdam.')


---
## What to say to the judges

> “The entry point is `run_stage2.py`. We begin with Madrid-trained Random Forest and LightGBM models.
> For this Amsterdam run we provide 200 labeled examples per age class. Our system adapts a local LDA and
> continues the LightGBM on those support labels. It then predicts 25,192 separate query pixels **without
> receiving their true ages**. After saving the predictions, we calculate Macro F1 against held-out development labels.
> Across ten 200-shot development trials, our mean Macro F1 was about **0.7232**. This is not an official hidden-test result.”

**If asked for a photo upload:** This submission works on **processed multispectral satellite measurements**, not standalone JPEG/PNG building photographs. The raw-to-feature preprocessing pipeline for arbitrary new imagery is **not yet verified**.

**Official submission:** Keep using the separately verified `BuildingAge_Final_Submission.zip` from `~/hackathon_results/presentation_day_submission_20261118/`. This notebook ZIP is a **demo companion**, not a replacement for the model-containing submission.
